# Bronze ~ Silver

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType

StatementMeta(, dfd461ff-f4ed-49b9-a8e7-d2aac3cb3c34, 3, Finished, Available, Finished, False)

In [2]:
# Read in bronze table
df = spark.read.table('bronze_online_retail')
# Check row count
print('Row Count:', df.count())

StatementMeta(, dfd461ff-f4ed-49b9-a8e7-d2aac3cb3c34, 4, Finished, Available, Finished, False)

Row Count: 541909


## Clean and Standardize

### Silver sales - Main transaction table

In [3]:
silver_sales = (
    df
    .withColumn('InvoiceNo', F.col('InvoiceNo').cast(StringType())) 
    .withColumn('StockCode', F.col('StockCode').cast(StringType()))
    .withColumn('Description', F.col('Description').cast(StringType()))
    .withColumn('Quantity', F.col('Quantity').cast(IntegerType()))
    .withColumn('UnitPrice', F.col('UnitPrice').cast(DoubleType()))
    .withColumn('CustomerID', F.col('CustomerID').cast(StringType()))
    .withColumn('Country', F.col('Country').cast(StringType()))
    .withColumn('InvoiceDate', F.to_timestamp('InvoiceDate', 'M/d/yy H:mm'))
    .filter(F.col('Quantity') > 0)
    .filter(F.col('UnitPrice') > 0)
    .filter(F.col('CustomerID').isNotNull())
    .filter(F.col('Description').isNotNull())
    .filter(~F.col('InvoiceNo').startswith('C'))
    .withColumn('SalesAmount', F.round(F.col('Quantity') * F.col('UnitPrice'), 2))
)

display(silver_sales)

StatementMeta(, dfd461ff-f4ed-49b9-a8e7-d2aac3cb3c34, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 360a112c-14d6-4fe7-afac-4b805b9e574c)

### Save the Silver sales table

In [4]:
silver_sales.write.mode('overwrite').saveAsTable('silver_sales')

StatementMeta(, dfd461ff-f4ed-49b9-a8e7-d2aac3cb3c34, 6, Finished, Available, Finished, False)

### Dimension tables

#### silver_products table - distinct stock code + description

In [5]:
silver_products = (
    silver_sales
    .select("StockCode", "Description")
    .dropDuplicates()
)

silver_products.write.mode("overwrite").saveAsTable("silver_products")
display(silver_products)

StatementMeta(, dfd461ff-f4ed-49b9-a8e7-d2aac3cb3c34, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fc92ee6a-d459-4e90-ae82-9741a96529ec)

#### silver_customers table - unique customers + associated country

In [6]:
silver_customers = (
    silver_sales
    .select("CustomerID", "Country")
    .dropDuplicates()
)

silver_customers.write.mode("overwrite").saveAsTable("silver_customers")
display(silver_customers)

StatementMeta(, dfd461ff-f4ed-49b9-a8e7-d2aac3cb3c34, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ec7c2d49-50be-409a-8ca6-2f6175ba465d)

#### silver_date table - year + quarter + month + month name + week + day

In [7]:
silver_date = (
    silver_sales
    .select(
        F.to_date("InvoiceDate").alias("Date")
    )
    .dropDuplicates()
    .withColumn("Year", F.year("Date"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("Month", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("WeekOfYear", F.weekofyear("Date"))
    .withColumn("Day", F.dayofmonth("Date"))
)

silver_date.write.mode("overwrite").saveAsTable("silver_date")
display(silver_date)

StatementMeta(, dfd461ff-f4ed-49b9-a8e7-d2aac3cb3c34, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 352fef63-8e34-47b5-96f0-d52e2ae5bb91)